# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

## stg_payment
- 데이터가 가벼워서 모든 컬럼 사용
- 금액(원화) 없음 → 매출액 분석 불가 > "구매 건수·판매 하트량" 기반으로 설계

## 특이사항
- payment_id, user_id, product_id, phone_type → 원본 보존 (식별·조인·재확인용)
- heart_amount → productId 파싱('heart.777'→777). **금액 컬럼 부재를 보완**하는 대체 규모 지표
- platform → I/A를 iOS/Android로 정규화
- purchased_at → created_at KST 변환 (타임존 통일)

## 타입 확인
- 원본: accounts_paymenthistory (id, user_id, productId, phone_type, created_at)
- created_at=TIMESTAMP (KST 변환 대상)

In [ ]:
%%bigquery df_pay_types --project your-gcp-project
SELECT column_name, data_type
FROM `your-gcp-project.raw.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'accounts_paymenthistory'
ORDER BY ordinal_position;

In [ ]:
df_pay_types

## 품질 및 기간 확인
- 가이드에 명시된 이벤트 데이터 대상 기간: 2023-07-18 ~ 2023-08-10 (약 4주) < Hackle에만 해당?
- 중복, 결측 없음

In [ ]:
%%bigquery df_pay_qc --project your-gcp-project
SELECT
  COUNT(*)                        AS total_payments,
  COUNT(*) - COUNT(DISTINCT id)   AS dup_id,           -- 0이어야 함
  COUNTIF(id IS NULL)             AS null_id,
  COUNTIF(user_id IS NULL)        AS null_user_id,
  COUNTIF(created_at IS NULL)     AS null_created_at,
  COUNTIF(productId IS NULL)      AS null_product,
  MIN(DATE(DATETIME(created_at,'Asia/Seoul'))) AS first_payment,
  MAX(DATE(DATETIME(created_at,'Asia/Seoul'))) AS last_payment
FROM `your-gcp-project.raw.accounts_paymenthistory`;

In [ ]:
df_pay_qc

## 상품 및 플랫폼
- 상품이 몇 종인지(하트 패키지 몇 개)를 봐야 어떤 상품이 잘 팔리는지 설계할 수 있음
- 금액은 없지만, 매출액 대신 판매된 하트 총량은 확인 가능
- heart.777이 1위인데 buyers가 거의 cnt랑 같음 = 1회성
- heart.1000은 cnt가 buyers의 약 2배 → 반복 구매
> 상품마다 구매 패턴이 상이함

In [ ]:
%%bigquery df_pay_product --project your-gcp-project
SELECT
  productId,
  phone_type,
  COUNT(*) AS cnt,
  COUNT(DISTINCT user_id) AS buyers
FROM `your-gcp-project.raw.accounts_paymenthistory`
GROUP BY productId, phone_type
ORDER BY cnt DESC;

In [ ]:
df_pay_product

## 테이블 생성
- heart_amount INT로 변환
- SAFE_CAST/SAFE_OFFSET은 형식이 다른 상품이 있어도 NULL로 빠지게 하는 방어장치

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE VIEW `your-gcp-project.stage.stg_payment` AS
WITH src AS (
  SELECT
    id, user_id, productId, phone_type, created_at
  FROM `your-gcp-project.raw.accounts_paymenthistory`
)
SELECT
  id                                 AS payment_id,
  user_id,
  productId                          AS product_id,     -- 원본 보존 (heart.777 등)
  -- 상품명에서 하트 수량 파싱: 'heart.777' → 777
  SAFE_CAST(SPLIT(productId, '.')[SAFE_OFFSET(1)] AS INT64) AS heart_amount,
  phone_type,
  -- 플랫폼 라벨 정규화
  CASE phone_type WHEN 'I' THEN 'iOS' WHEN 'A' THEN 'Android' END AS platform,
  DATETIME(created_at, 'Asia/Seoul') AS purchased_at    -- 구매시각 (KST)
FROM src;

### 생성 검증

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*)                          AS total_payments,   -- 95,140
  COUNTIF(heart_amount IS NULL)     AS parse_fail,        -- 0이어야 (파싱 실패 = 예상 못한 상품명)
  COUNT(DISTINCT product_id)        AS product_types,     -- 상품 종류 수 (4개?)
  COUNTIF(platform IS NULL)         AS platform_null,     -- 0이어야 (I/A 외 값 있으면 >0)
  SUM(heart_amount)                 AS total_hearts_sold  -- 판매된 하트 총량
FROM `your-gcp-project.stage.stg_payment`;